In [ ]:
path = r'kc_house_data.csv'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv(path)
df.head()

In [ ]:
df.columns

### EDA

In [ ]:
df.isnull().sum()

In [ ]:
sns.histplot(df['price'])

In [ ]:
sns.countplot(df['bedrooms'])

In [ ]:
sns.scatterplot(x='price', y='sqft_living', data=df)

In [ ]:
sns.scatterplot(x='long', y='price', data=df)

In [ ]:
sns.scatterplot(x='lat', y='price', data=df)

In [ ]:
sns.scatterplot(x='lat', y='long', data=df, hue='price')

In [ ]:
sns.boxplot(x='waterfront', y='price', data=df)

### Prepare the Data

In [ ]:
df.head()

In [ ]:
df = df.drop('id', axis=1)

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df.head()

In [ ]:
df['month'] = df['date'].apply(lambda d : d.month)
df['year'] = df['date'].apply(lambda d : d.year)
df.head()

In [ ]:
df.groupby('month').mean()['price'].plot()

In [ ]:
df.groupby('year').mean()['price'].plot()

In [ ]:
df = df.drop('date',axis=1)

### Training

In [ ]:
#!pip install scikit-learn tensorflow

In [ ]:
import tensorflow as tf
from tensorflow import keras
print(tf.__version__)
print(keras.__version__)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation
from tensorflow.keras.optimizers import Adam

In [ ]:
X = df.drop('price',axis=1)
y = df['price']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state=101)

### Scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_train= scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Model Creation

In [ ]:
model = Sequential()

model.add(Dense(19,activation='relu'))
model.add(Dense(19,activation='relu'))
model.add(Dense(19,activation='relu'))
model.add(Dense(19,activation='relu'))
model.add(Dense(1))

model.compile(optimizer='adam',loss='mse')

In [ ]:
model.fit(
    x=X_train,
    y=y_train.values,
    validation_data=(X_test,y_test.values),
    batch_size=128,
    epochs=400
)

In [ ]:
losses = pd.DataFrame(model.history.history)

In [ ]:
losses.plot()

### Evaluation

In [ ]:
from sklearn.metrics import mean_squared_error,mean_absolute_error,explained_variance_score

In [ ]:
predictions = model.predict(X_test)

In [ ]:
mean_absolute_error(y_test,predictions)

In [ ]:
np.sqrt(mean_squared_error(y_test,predictions))


In [ ]:
explained_variance_score(y_test,predictions)

In [ ]:
df['price'].mean()

In [ ]:
# Our predictions
plt.scatter(y_test,predictions)

# Perfect predictions
plt.plot(y_test,y_test,'r')

### Predicting for brand new house

In [ ]:
df.head()

In [ ]:
single_house = df.drop('price', axis=1).iloc[0]
single_house.shape

In [ ]:
single_house = scaler.transform(single_house.values.reshape(-1, 20))

In [ ]:
single_house

In [ ]:
model.predict(single_house)